In [ ]:
import torch.nn.functional as F
import torch
import torch.nn as nn
import imageio as io
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as transforms
import random

In [ ]:
def plot(x):
    fig, ax = plt.subplots()
    im = ax.imshow(x, cmap = 'gray')
    ax.axis('off')
    fig.set_size_inches(18, 10)
    plt.show()

In [ ]:
def montage(images_tensor):
    b, c, h, w = images_tensor.shape
    assert c == 1 or c == 3, "Number of channels must be either 1 (grayscale) or 3 (RGB)"

    montage_w = int(np.ceil(np.sqrt(b)))
    montage_h = int(np.ceil(b / montage_w))

    if c == 1:
        montage_image = np.zeros((montage_h * h, montage_w * w))
    else:
        montage_image = np.zeros((montage_h * h, montage_w * w, c))

    for idx, img in enumerate(images_tensor):
        i = idx // montage_w
        j = idx % montage_w
        if c == 1:
            montage_image[i * h:(i + 1) * h, j * w:(j + 1) * w] = img.squeeze(0)
        else:
            montage_image[i * h:(i + 1) * h, j * w:(j + 1) * w, :] = img.permute(1, 2, 0)

    return montage_image

In [ ]:
def scale1(x):
    return (x-np.min(x))/(np.max(x)-np.min(x))

In [ ]:
def process(image):
    image = np.mean(image,axis=2)
    image = scale1(image)
    image = image - np.mean(image)
    image = torch.from_numpy(image)
    image = image[None,:,:]
    return image

In [ ]:
def plot_map(z,color='red',thresh=3):
    z == torch.max(z)
    [y,x] = torch.where(torch.abs(z - torch.amax(z)) <  thresh)
    fig, ax = plt.subplots()
    im = ax.imshow(image[0,0], cmap = 'gray')
    ax.axis('off')
    fig.set_size_inches(18, 10)
    ax.scatter(x+7, y+7, c=color, s=40)
    plt.show()
    # print(x,y)

In [ ]:
image = io.imread("http://ian-albert.com/games/super_mario_bros_maps/mario-2-2.gif")

image = image[:,:,:]
image0 = image.copy()

mario = image[31:46,41:56,:]
jelly = image[152:167,312:327,:]
coin = image[185:200,224:239,:]
bug1 = image[105:120,1179:1194,:]
bug2 = image[80:95,1264:1279,:]

In [ ]:
plot(image)
plot(mario)
plot(jelly)
plot(coin)
plot(bug1)
plot(bug2)

In [ ]:
image = process(image)
mario = process(mario)
coin  = process(coin)
jelly = process(jelly)
bug1  = process(bug1)
bug2  = process(bug2)

In [ ]:
objects = torch.stack([mario,coin,jelly,bug1,bug2])
image   = image[None,:,:]

In [ ]:
objects.shape

In [ ]:
maps = F.conv2d(image,objects)

In [ ]:
plot(image0)
plot_map(maps[0,0],'red',0.1)
plot_map(maps[0,1],'yellow')
plot_map(maps[0,2],'blue')
plot_map(maps[0,3],'green')
plot_map(maps[0,4],'magenta')

#Random Template

In [ ]:
image = io.imread("http://ian-albert.com/games/super_mario_bros_maps/mario-2-2.gif")

image = image[:,:,:]
image0 = image.copy()

mario = image[31:46,41:56,:]

In [ ]:
mario.shape,image.shape

In [ ]:
s = mario.shape[1]

In [ ]:
rx = np.random.randint(image.shape[0]-15)
ry = np.random.randint(image.shape[1]-15)

In [ ]:
# (27, 2715)

In [ ]:
rx,ry

In [ ]:
patch = image[rx:rx+s,ry:ry+s,:]

In [ ]:
plot(patch)

In [ ]:
image = process(image)
patch = process(patch)

In [ ]:
objects = torch.stack([patch])
image   = image[None,:,:]

In [ ]:
maps = F.conv2d(image,objects)

In [ ]:
plot(image0)
plot_map(maps[0,0],'red',3)

#Entropy Patch Measure

In [ ]:
machine_epsilon = np.finfo(float).eps

In [ ]:
machine_epsilon

In [ ]:
def entropy(image):

    gray_image = np.mean(image,2)

    normalized_image = gray_image / 255.0

    hist, _ = np.histogram(normalized_image, bins=256, range=(0, 1))

    total_pixels = gray_image.size
    probabilities = hist / total_pixels

    entropy = -np.sum(probabilities * np.log2(probabilities + machine_epsilon))

    return entropy

## View as Windows

In [ ]:
def view_as_windows_torch(image, shape, stride=None):

    if stride is None:
        stride = shape[0] // 2, shape[1] // 2

    windows = image.unfold(2, shape[0], stride[0])
    return windows.unfold(3, shape[1], stride[1])

In [ ]:
x = torch.arange(16).reshape((1, 1, 4, 4))
patches = view_as_windows_torch(image=x, shape=(2, 2))

In [ ]:
patches.shape

# Image Entropy Batch Version

In [ ]:
def batch_image_entropy(images):

    # Normalize pixel intensities
    normalized_images = images / 255.0

    # Initialize the entropies tensor
    entropies = torch.zeros(images.shape[0])

    # Calculate the entropy for each image in the batch
    for i, image in enumerate(normalized_images):
        hist = torch.histc(image, bins=256, min=0, max=1)
        probabilities = hist / (images.shape[2] * images.shape[3])
        entropies[i] = -torch.sum(probabilities * torch.log2(probabilities + torch.finfo(float).eps))

    return entropies

# Example usage
B, C, H, W = 8, 1, 64, 64  # Batch size, channels, height, width
images = torch.randn(B, C, H, W)  # Random grayscale images
entropies = batch_image_entropy(images)
print('Entropies:', entropies)


# Random Filters Experiment

In [ ]:
image = io.imread("http://ian-albert.com/games/super_mario_bros_maps/mario-2-2.gif")

image = image[:,:,:]
image = process(image)[None,:,:,:]

s = 15

In [ ]:
image.shape

In [ ]:
patches = view_as_windows_torch(image, (s, s))[0,0]

In [ ]:
a,b,c,d = patches.shape

In [ ]:
patches = patches.reshape(a * b, 1, c, d)

In [ ]:
patches.shape

In [ ]:
entropies = batch_image_entropy(patches)

In [ ]:
entropies

In [ ]:
sort = torch.argsort(entropies).flip(0)

In [ ]:
sort = sort[0:25]

In [ ]:
objects = patches[sort]

In [ ]:
objects.shape

In [ ]:
montage_img = montage(objects)
plt.imshow(montage_img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
def scale2(x):
    return (x - x.min()) / (x.max() - x.min())

def process2(image):
    image = torch.mean(image, dim=1, keepdim=True)
    image = scale2(image)
    image = image - torch.mean(image)
    return image


In [ ]:
objects = process2(objects)

In [ ]:
patches.shape,image.shape

In [ ]:
maps = F.conv2d(image,objects)

In [ ]:
maps.shape

In [ ]:
plot(image0)

In [ ]:
def plot_map(z, thresh=3):
    z == torch.max(z)
    [y, x] = torch.where(torch.abs(z - torch.amax(z)) <  thresh)
    fig, ax = plt.subplots()
    im = ax.imshow(image[0, 0], cmap='gray')
    ax.axis('off')
    fig.set_size_inches(18, 10)

    # Generate a random RGB color
    color = (random.random(), random.random(), random.random())

    ax.scatter(x + 7, y + 7, color=color, s=40)
    plt.show()

In [ ]:
for i in range(maps.shape[1]):
    plot_map(maps[0,i],5)